# Recommender System Exploration and Analysis

Import and NoteBook Setup

In [ ]:
#install repo library
!pip install kagglehub

In [ ]:
# Import initial 3rd party libraries
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import json

# Configure Notebook
# %matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_context("notebook")
# Force white background after fivethirtyeight
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
sns.set_style("whitegrid")        # or "white"
sns.despine(left=True, bottom=True)
import warnings
warnings.filterwarnings('ignore')

### Load the Dataset

In [ ]:
# Download latest version
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("Path to dataset files:", path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(f"Directory: {root}")
    for file in files:
        file_path = os.path.join(root, file)
        print(f"  File: {file_path}")
        print(file)
        

### Load the files and do basic cleaning  
To run with the large dataset, remove the '_small' from the ratings and links file string

In [ ]:
# Load the metadata
# data_path = "../../datasets/input/"
data_path = path+"\\"
df_meta = pd.read_csv(data_path+"movies_metadata.csv")
df_meta = df_meta[["id", "title"]]
df_meta["id"] = pd.to_numeric(df_meta["id"], errors="coerce") 
df_meta = df_meta.dropna(subset=["id"]).astype({"id": int}) 
df_meta = df_meta.dropna(subset=["title"]) 
df_meta = df_meta.drop_duplicates(subset=['id'],keep='first')

# Load the ratings dataset
df_ratings = pd.read_csv(data_path+"ratings_small.csv")
df_ratings = df_ratings[["userId", "movieId", "rating"]]
df_ratings = df_ratings.dropna(subset=["userId", "movieId", "rating"])
df_ratings["userId"] = df_ratings["userId"].astype(int) 
df_ratings["movieId"] = df_ratings["movieId"].astype(int) 
df_ratings["rating"] = df_ratings["rating"].astype(float) 
# If there are duplicate <userId, movieId> pairs, average them. Unlikely though 
df_ratings = df_ratings.groupby(["userId", "movieId"], as_index=False)["rating"].mean()


df_links = pd.read_csv(data_path+"links_small.csv") 
df_links = df_links[["movieId", "tmdbId"]]
df_links["tmdbId"] = pd.to_numeric(df_links["tmdbId"], errors="coerce") 
df_links = df_links.dropna(subset=["tmdbId"]).astype({"tmdbId": int}) 

### Filter outliers and users who have few ratings  
These users can skew the data and introduce noise in the system

In [ ]:
user_counts = df_ratings['userId'].value_counts().sort_index()

plot_df = pd.DataFrame({
    'userId': user_counts.index,
    'rating_count': user_counts.values
})

In [ ]:
sns.scatterplot(plot_df,x='userId',y='rating_count')

In [ ]:
# Filter users who have rated too many or too few movies
print("="*50)
print("FILTER USERS")
print("="*50)

# Count how many movies each user has rated
user_movie_counts = df_ratings['userId'].value_counts()
print(f"User rating counts - Min: {user_movie_counts.min()}, Max: {user_movie_counts.max()}")

# Remove users who have rated less than 5 movies (10  for full dataset)
min_user_ratings = 5
users_to_keep = user_movie_counts[user_movie_counts >= min_user_ratings].index
print(f"Users with at least {min_user_ratings} ratings: {len(users_to_keep)} out of {len(user_movie_counts)}")

# Remove users who rated too many movies (top 5% likely outliers)
max_percentile = 95
max_user_ratings = np.percentile(user_movie_counts, max_percentile)
users_to_keep = users_to_keep[user_movie_counts[users_to_keep] <= max_user_ratings]
print(f"Users after removing top 5% (>{max_user_ratings:.0f} ratings): {len(users_to_keep)}")

# Filter the dataset to keep only selected users
ratings_filtered = df_ratings[df_ratings['userId'].isin(users_to_keep)].copy()
print(f"Ratings after user filtering: {ratings_filtered.shape}")


### Filter movies with few ratings 
These movies introduce noise in the system and increase computation memory

In [ ]:
movie_counts = df_ratings['movieId'].value_counts().sort_index()

plot_df = pd.DataFrame({
    'movieId': movie_counts.index,
    'rating_count': movie_counts.values
})

In [ ]:
sns.scatterplot(plot_df,x='movieId',y='rating_count')

In [ ]:
# Filter movies with less than 20 ratings

print("="*50)
print("FILTERING MOVIES")
print("="*50)

# Count how many ratings each movie has received
movie_rating_counts = ratings_filtered['movieId'].value_counts()
print(f"Movie rating counts - Min: {movie_rating_counts.min()}, Max: {movie_rating_counts.max()}")

# Keep only movies with at least 20 ratings
min_movie_ratings = 20
movies_to_keep = movie_rating_counts[movie_rating_counts >= min_movie_ratings].index
print(f"Movies with at least {min_movie_ratings} ratings: {len(movies_to_keep)} out of {len(movie_rating_counts)}")

# Filter the dataset to keep only selected movies
ratings_final = ratings_filtered[ratings_filtered['movieId'].isin(movies_to_keep)].copy()
print(f"Final ratings shape: {ratings_final.shape}")


Rating distribution 

In [ ]:
sns.countplot(x="rating", data=ratings_final, palette="magma")
plt.title("Distribution of movie ratings", fontsize=14)
plt.show()


### Create the index mappings  
Since users and movies were removed they are no longer sequential and have gaps.
The mappings will keep track of the real id's after the computations

In [ ]:
# Create index mappings
print("="*50)
print("CREATING INDEX MAPPINGS")
print("="*50)

unique_users = sorted(ratings_final['userId'].unique())
unique_movies = sorted(ratings_final['movieId'].unique())

print(f"Final number of users: {len(unique_users)}")
print(f"Final number of movies: {len(unique_movies)}")

user_id_to_index = {user_id: idx for idx, user_id in enumerate(unique_users)}
index_to_user_id = {idx: user_id for user_id, idx in user_id_to_index.items()}
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(unique_movies)}
index_to_movie_id = {idx: movie_id for movie_id, idx in movie_id_to_index.items()}

ratings_final['user_idx'] = ratings_final['userId'].map(user_id_to_index)
ratings_final['movie_idx'] = ratings_final['movieId'].map(movie_id_to_index)

### Create the sparse User-Item matrix

In [ ]:
# Create the user-movie rating matrix
print("="*50)
print("CREATING USER-MOVIE MATRIX")
print("="*50)

from scipy.sparse import csr_matrix

n_users = len(unique_users)
n_movies = len(unique_movies)

# Convert to numpy arrays for faster indexing
user_indices = ratings_final['user_idx'].values.astype(int)
movie_indices = ratings_final['movie_idx'].values.astype(int)
rating_values = ratings_final['rating'].values.astype(float)

# Create a sparse user-movie matrix using csr_matrix
# Data, (row_indices, col_indices)
user_movie_matrix = csr_matrix((rating_values, (user_indices, movie_indices)), shape=(n_users, n_movies))

print(f"User-movie matrix shape: {user_movie_matrix.shape}")
# Sparsity for csr_matrix can be calculated as 1 - (number of stored elements / total possible elements)
print(f"Matrix sparsity: {(1 - user_movie_matrix.nnz / (n_users * n_movies)) * 100:.2f}%")

### Load surprise library for model analysis

In [ ]:
from surprise import Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split, GridSearchCV
from surprise import SVD, NMF, CoClustering, KNNBasic, KNNWithMeans, KNNWithZScore, SlopeOne
import pandas as pd
import numpy as np

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_final[['userId', 'movieId', 'rating']], reader)


### Run 5 cross-fold validation on each model to check initial performance


In [ ]:
# SVD

algo=svd= SVD()
svd_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)


In [ ]:
# User based kNN

algo=kNNBasic_user= KNNBasic(sim_options={'user_based': True})
kNNBasicUser_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

In [ ]:
# Item based kNN

algo=kNNBasic_item= KNNBasic(sim_options={'user_based': False})
kNNBasicItem_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

Graph the results

In [ ]:
dplot = {
    'SVD':svd_cv_results,
    'User CF':kNNBasicUser_cv_results,
    'Item CF':kNNBasicItem_cv_results,
}
df_list = []

for model_name, result in dplot.items():
    dfdl = pd.DataFrame({
        'test_rmse': result['test_rmse'],
        'test_mae':  result['test_mae'],
        'fit_time':  result['fit_time'],
        'test_time': result['test_time']
    })
    dfdl['model'] = model_name
    df_list.append(dfdl)

# Combine all folds from all models
df_all = pd.concat(df_list, ignore_index=True)

# Reorder columns
df_all = df_all[['model', 'test_rmse', 'test_mae', 'fit_time', 'test_time']]

print(df_all.round(5))

In [ ]:
# Calc average RMSE and MAE per model ---
means = df_all.groupby('model')[['test_rmse', 'test_mae']].mean().round(4)
means = means.reset_index()

# Melt to long format for seaborn plot
plot_df = means.melt(id_vars='model',
                     value_vars=['test_rmse', 'test_mae'],
                     var_name='Metric',
                     value_name='Score')

# Make names nicer
plot_df['Metric'] = plot_df['Metric'].replace({
    'test_rmse': 'RMSE',
    'test_mae':  'MAE'
})

# Order models
model_order = ['User CF', 'Item CF', 'SVD']  # or whatever order you prefer
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)
plot_df = plot_df.sort_values('model')


# Plot
plt.figure(figsize=(9, 6))
sns.set_context("notebook", font_scale=1.3)
ax = sns.barplot(data=plot_df,
                 x='model',
                 y='Score',
                 hue='Metric',
                 palette='magma')

# Add values on top of each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0.01:
            ax.text(p.get_x() + p.get_width()/2.,
                    height + 0.008,
                    f'{height:.4f}',
                    ha='center', va='bottom',
                    fontweight='bold', fontsize=12)

ax.set_title('Average RMSE & MAE From a 5-fold CV\nSmall Dataset',
             fontsize=17, pad=20)
ax.set_ylabel('Error Score', fontsize=13)
ax.set_xlabel('Model', fontsize=13)
ax.legend(title='Metric', title_fontsize=12, fontsize=11, loc='upper right')

plt.ylim(0, 1.0)
sns.despine(trim=True)
plt.tight_layout()
plt.show()

### Try with taking into account user and movie mean ratings
This should balance the data and deal with users who rate high or low all the time  
Should also deal with popular movies to offer more unique selections

In [ ]:
# User based kNN taking into account the mean ratings of each user

algo = kNNWithMeans_user= KNNWithMeans(sim_options={'user_based': True})
kNNMeansUser_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

In [ ]:
# Item based kNN taking into account the mean ratings of each user

algo=kNNWithMeans_item= KNNWithMeans(sim_options={'user_based': False})
kNNMeansItem_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

Make the plots again  
Use line graph as bars get hard to see when the values are close

In [ ]:
dplot = {
    'SVD':      svd_cv_results,
    'User CF':kNNBasicUser_cv_results,
    'Item CF':kNNBasicItem_cv_results,
    'User CF Mean':kNNMeansUser_cv_results,
    'Item CF Mean':kNNMeansItem_cv_results
}

df_list = []
for model_name, result in dplot.items():
    df_temp = pd.DataFrame({
        'test_rmse': result['test_rmse'],
        'test_mae':  result['test_mae']
    })
    df_temp['model'] = model_name
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)

# Calc average per model
means = df_all.groupby('model')[['test_rmse', 'test_mae']].mean().round(4)
means = means.reset_index()

# Melt to long format again
plot_df = means.melt(id_vars='model',
                     value_vars=['test_rmse', 'test_mae'],
                     var_name='Metric',
                     value_name='Score')

# Clean names
plot_df['Metric'] = plot_df['Metric'].replace({'test_rmse': 'RMSE', 'test_mae': 'MAE'})

# Order models
model_order = ['User CF', 'Item CF','User CF Mean', 'Item CF Mean', 'SVD']
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)
plot_df = plot_df.sort_values('model')

# Use a line graph this time
plt.figure(figsize=(9, 6))
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.3)

# Lineplot with markers
ax = sns.lineplot(data=plot_df,
                  x='model',
                  y='Score',
                  hue='Metric',
                  marker='o',
                  markersize=14,
                  palette='magma')

# Add values next to each point
for _, row in plot_df.iterrows():
    ax.text(row['model'], 
            row['Score'] + 0.008,
            f'{row["Score"]:.4f}',
            ha='center', va='bottom',
            fontweight='bold', fontsize=12, color='black')

ax.set_title('Average RMSE & MAE From a 5-fold CV\nSmall Dataset',
             fontsize=17, pad=20)
ax.set_ylabel('Error Score', fontsize=13)
ax.set_xlabel('Model', fontsize=13)
ax.legend(title='Metric', title_fontsize=12, fontsize=11, loc='upper right')

plt.ylim(0.60, 0.95)
sns.despine(trim=True)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Another line plot, but with some metrics from the full dataset  
Manually entered since cross-fold valuation on the full set takes a long time. Values came from the full dataset run

In [ ]:
dplot = {
    'SVD':      svd_cv_results,
    'SVD full':      {
        'test_rmse': np.array([0.8410 , 0.8415 , 0.8417,  0.8397,  0.8417]),
        'test_mae':  np.array([ 0.6398,  0.6404 , 0.6407,  0.6393,  0.6406]),
        'fit_time':  (  56.81,   64.98,   66.71,   66.90 ,  66.79),
        'test_time': ( 361.53,  26.48 ,  31.38 ,  21.22,   19.34 )
        },
    'User CF':kNNBasicUser_cv_results,
    'Item CF':kNNBasicItem_cv_results,
    'Item CF full':{
        'test_rmse': np.array([0.8971,  0.8974,  0.8963,  0.8968,  0.8967]),
        'test_mae':  np.array([ 0.6813,  0.6814,  0.6808,  0.6812,  0.6811 ]),
        'fit_time':  ( 33.35,   35.85,   34.41,   33.33,   34.83),
        'test_time': ( 139.70,  201.47,  150.90,  161.40,  156.74 )
        },
    'User CF Mean':kNNMeansUser_cv_results,
    'Item CF Mean':kNNMeansItem_cv_results,
    'Item CF Mean full':{
        'test_rmse': np.array([0.8587,  0.8589,  0.8591,  0.8590,  0.8599]),
        'test_mae':  np.array([ 0.6525,  0.6525,  0.6525,  0.6525,  0.6533]),
        'fit_time':  ( 34.41,   37.14,   37.16,   36.54,   32.35),
        'test_time': (461.38,  178.71,  241.97, 389.37,  279.53 )
        }
}

df_list = []
for model_name, result in dplot.items():
    df_temp = pd.DataFrame({
        'test_rmse': result['test_rmse'],
        'test_mae':  result['test_mae']
    })
    df_temp['model'] = model_name
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)

# Calc average per model
means = df_all.groupby('model')[['test_rmse', 'test_mae']].mean().round(4)
means = means.reset_index()

# Melt to long format
plot_df = means.melt(id_vars='model',
                     value_vars=['test_rmse', 'test_mae'],
                     var_name='Metric',
                     value_name='Score')

# Clean names
plot_df['Metric'] = plot_df['Metric'].replace({'test_rmse': 'RMSE', 'test_mae': 'MAE'})

# Order models
model_order = ['User CF', 'Item CF','Item CF full','User CF Mean', 'Item CF Mean', 'Item CF Mean full', 'SVD', 'SVD full']  # or whatever order you prefer
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)
plot_df = plot_df.sort_values('model')

plt.figure(figsize=(14, 6))
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.3)

# Lineplot with markers
ax = sns.lineplot(data=plot_df,
                  x='model',
                  y='Score',
                  hue='Metric',
                  marker='o',
                  markersize=14,
                  palette='magma')

# Add values next to each point
for _, row in plot_df.iterrows():
    ax.text(row['model'], 
            row['Score'] + 0.008,
            f'{row["Score"]:.4f}',
            ha='center', va='bottom',
            fontweight='bold', fontsize=12, color='black')

ax.set_title('Average RMSE & MAE From a 5-fold CV\nSmall and Large Dataset',
             fontsize=17, pad=20)
ax.set_ylabel('Error Score', fontsize=13)
ax.set_xlabel('Model', fontsize=13)
ax.legend(title='Metric', title_fontsize=12, fontsize=11, loc='upper right')

plt.ylim(0.60, 0.95)  
sns.despine(trim=True)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Graph the fit times of the full dataset. The small dataset is commented out. Uncomment them to include  
If graphing the small dataset fit times it is a good idea not to include the full dataset times

In [ ]:
dplot = {
    # 'SVD':      svd_cv_results,
    'SVD full':      {
        'test_rmse': np.array([0.8410 , 0.8415 , 0.8417,  0.8397,  0.8417]),
        'test_mae':  np.array([ 0.6398,  0.6404 , 0.6407,  0.6393,  0.6406]),
        'fit_time':  (  56.81,   64.98,   66.71,   66.90 ,  66.79),
        'test_time': ( 361.53,  26.48 ,  31.38 ,  21.22,   19.34 )
        },
    # 'User CF':kNNBasicUser_cv_results,
    # 'Item CF':kNNBasicItem_cv_results,
    'Item CF full':{
        'test_rmse': np.array([0.8971,  0.8974,  0.8963,  0.8968,  0.8967]),
        'test_mae':  np.array([ 0.6813,  0.6814,  0.6808,  0.6812,  0.6811 ]),
        'fit_time':  ( 33.35,   35.85,   34.41,   33.33,   34.83),
        'test_time': ( 139.70,  201.47,  150.90,  161.40,  156.74 )
        },
    # 'User CF Mean':kNNMeansUser_cv_results,
    # 'Item CF Mean':kNNMeansItem_cv_results,
    'Item CF Mean full':{
        'test_rmse': np.array([0.8587,  0.8589,  0.8591,  0.8590,  0.8599]),
        'test_mae':  np.array([ 0.6525,  0.6525,  0.6525,  0.6525,  0.6533]),
        'fit_time':  ( 34.41,   37.14,   37.16,   36.54,   32.35),
        'test_time': (461.38,  178.71,  241.97, 389.37,  279.53 )
        }
}

df_list = []
for model_name, result in dplot.items():
    df_temp = pd.DataFrame({
        'fit_time': result['fit_time'],
        'test_time':  result['test_time']
    })
    df_temp['model'] = model_name
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)

# Calc average per model
means = df_all.groupby('model')[['fit_time', 'test_time']].mean().round(4)
means = means.reset_index()

# Melt to long format
plot_df = means.melt(id_vars='model',
                     value_vars=['fit_time', 'test_time'],
                     var_name='Metric',
                     value_name='Time')

# Clean names
plot_df['Metric'] = plot_df['Metric'].replace({'fit_time': 'Fit time', 'test_time': 'Test time'})

# model_order = ['User CF', 'Item CF','User CF Mean', 'Item CF Mean', 'SVD'] 
model_order = ['Item CF full', 'Item CF Mean full', 'SVD full'] 
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)
plot_df = plot_df.sort_values('model')

plt.figure(figsize=(9, 6))
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.3)

# Lineplot with markers
ax = sns.lineplot(data=plot_df,
                  x='model',
                  y='Time',
                  hue='Metric',
                  marker='o',
                  markersize=14,
                  palette='magma')

# Add values next to each point
for _, row in plot_df.iterrows():
    ax.text(row['model'], 
            row['Time'] + 0.008,
            f'{row["Time"]:.4f}',
            ha='center', va='bottom',
            fontweight='bold', fontsize=12, color='black')

ax.set_title('Average Fit & Test times From a 5-fold CV\nLarge Dataset',
             fontsize=17, pad=20)
ax.set_ylabel('Time (s)', fontsize=13)
ax.set_xlabel('Model', fontsize=13)
ax.legend(title='Metric', title_fontsize=12, fontsize=11, loc='upper right')

plt.ylim(0, 320)
sns.despine(trim=True)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Attempted with z-score, but found little improvement

In [ ]:
# User based kNN taking into account the z-score normalization of each user

algo=kNNWithZScore_user= KNNWithZScore(sim_options={'user_based': True})
kNNzUser_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

In [ ]:
# Item based kNN taking into account the z-score normalization of each user

algo=kNNWithZScore_item= KNNWithZScore(sim_options={'user_based': False})
kNNzItem_cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

## Hypertuning

In [ ]:
# Tuning

# SVD
# grid search
print("SVD Grid Search")

param_grid = {
    "n_factors":[50,100,150,200],
    "n_epochs": [5, 10, 15, 20, 25,30], 
    "lr_all": [0.002, 0.005, 0.007]
    }

grid_search = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3)
grid_search.fit(data)
svd_algo = grid_search.best_estimator["rmse"]

svd_results_df = pd.DataFrame.from_dict(grid_search.cv_results)
svd_results_df.head()


In [ ]:
svd_results_df.sort_values('mean_test_rmse').head()

In [ ]:
svd_cv_results_h = cross_validate(svd_algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

Plot parameter change effects

In [ ]:
sns.lineplot(svd_results_df,x='param_n_factors', y='mean_test_rmse')

In [ ]:
sns.lineplot(svd_results_df,x='param_n_epochs', y='mean_test_rmse')

In [ ]:
sns.lineplot(svd_results_df,x='param_lr_all', y='mean_test_rmse')

In [ ]:
# kNN Basic User
# grid search
print("kNN Basic User Grid Search")

param_grid = {

    "k":[20,30,40,60,70],
    "min_k": [1,2,3,4],
    'sim_options': {
        'name': ['msd', 'cosine','pearson'],
        'user_based': [True],
    },
    'verbose':[False]
    }
grid_search = GridSearchCV(KNNBasic, param_grid, measures=["rmse"], cv=3)
grid_search.fit(data)
kNNBasicUser_algo = grid_search.best_estimator["rmse"]
# algo.fit(data.build_full_trainset())

kNNBasicUser_results_df = pd.DataFrame.from_dict(grid_search.cv_results)
kNNBasicUser_results_df.head()

In [ ]:
kNNBasicUser_results_df.sort_values('mean_test_rmse').head()

In [ ]:
sns.lineplot(kNNBasicUser_results_df,x='param_k', y='mean_test_rmse')

In [ ]:
sns.lineplot(kNNBasicUser_results_df,x='param_min_k', y='mean_test_rmse')

In [ ]:
kNNBasicUser_results_df2=kNNBasicUser_results_df.join(pd.DataFrame(kNNBasicUser_results_df.pop('param_sim_options').values.tolist()))

In [ ]:
sns.lineplot(kNNBasicUser_results_df2,x='name', y='mean_test_rmse')

In [ ]:
# kNN Basic Item
# grid search
print("kNN Basic Item Grid Search")

param_grid = {

    "k":[20,30,40,60,70],
    "min_k": [1,2,3,4],
    'sim_options': {
        'name': ['msd', 'cosine','pearson'],
        'user_based': [False],
    },
    'verbose':[False]
    }
grid_search = GridSearchCV(KNNBasic, param_grid, measures=["rmse"], cv=3)
grid_search.fit(data)
kNNBasicItem_algo = grid_search.best_estimator["rmse"]
# algo.fit(data.build_full_trainset())

kNNBasicItem_results_df = pd.DataFrame.from_dict(grid_search.cv_results)
kNNBasicItem_results_df.head()

In [ ]:
kNNBasicItem_results_df.sort_values('mean_test_rmse').head()

In [ ]:
sns.lineplot(kNNBasicItem_results_df,x='param_k', y='mean_test_rmse')

In [ ]:
sns.lineplot(kNNBasicItem_results_df,x='param_min_k', y='mean_test_rmse')

In [ ]:
kNNBasicItem_results_df2=kNNBasicItem_results_df.join(pd.DataFrame(kNNBasicItem_results_df.pop('param_sim_options').values.tolist()))

In [ ]:
sns.lineplot(kNNBasicItem_results_df2,x='name', y='mean_test_rmse')

In [ ]:
# kNN Means User
# grid search
print("kNN Means User Grid Search")

param_grid = {

    "k":[20,30,40,60,70],
    "min_k": [1,2,3,4],
    'sim_options': {
        'name': ['msd', 'cosine','pearson'],
        'user_based': [True],
    },
    'verbose':[False]
    }
grid_search = GridSearchCV(KNNWithMeans, param_grid, measures=["rmse"], cv=3)
grid_search.fit(data)
kNNMeansUser_algo = grid_search.best_estimator["rmse"]
# algo.fit(data.build_full_trainset())

kNNMeansUser_results_df = pd.DataFrame.from_dict(grid_search.cv_results)
kNNMeansUser_results_df.head()

In [ ]:
kNNMeansUser_cv_results_h = cross_validate(kNNMeansUser_algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

In [ ]:
# pd.set_option('display.width', 500)
pd.set_option('display.max_colwidth', None)
kNNMeansUser_results_df.sort_values('mean_test_rmse')['params'].head()

In [ ]:
sns.lineplot(kNNMeansUser_results_df,x='param_k', y='mean_test_rmse')

In [ ]:
sns.lineplot(kNNMeansUser_results_df,x='param_min_k', y='mean_test_rmse')

In [ ]:
kNNMeansUser_results_df2=kNNMeansUser_results_df.join(pd.DataFrame(kNNMeansUser_results_df.pop('param_sim_options').values.tolist()))

In [ ]:
sns.lineplot(kNNMeansUser_results_df2,x='name', y='mean_test_rmse')

In [ ]:
# kNN Means Item
# grid search
print("kNN Means Item Grid Search")

param_grid = {

    "k":[20,30,40,60,70],
    "min_k": [1,2,3,4],
    'sim_options': {
        'name': ['msd', 'cosine','pearson'],
        'user_based': [False],
    },
    'verbose':[False]
    }
grid_search = GridSearchCV(KNNWithMeans, param_grid, measures=["rmse"], cv=3)
grid_search.fit(data)
kNNMeansItem_algo = grid_search.best_estimator["rmse"]
# algo.fit(data.build_full_trainset())

kNNMeansItem_results_df = pd.DataFrame.from_dict(grid_search.cv_results)
kNNMeansItem_results_df.head()

In [ ]:
kNNMeansItem_results_df.sort_values('mean_test_rmse').head()

In [ ]:
kNNMeansItem_cv_results_h = cross_validate(kNNMeansItem_algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True, n_jobs=1)

In [ ]:
sns.lineplot(kNNMeansItem_results_df,x='param_k', y='mean_test_rmse')

In [ ]:
sns.lineplot(kNNMeansItem_results_df,x='param_min_k', y='mean_test_rmse')

In [ ]:
kNNMeansItem_results_df2=kNNMeansItem_results_df.join(pd.DataFrame(kNNMeansItem_results_df.pop('param_sim_options').values.tolist()))

In [ ]:
sns.lineplot(kNNMeansItem_results_df2,x='name', y='mean_test_rmse')

### Graph tuned results

In [ ]:
dplot = {
    'SVD BH':      svd_cv_results,
    'SVD AH':      svd_cv_results_h,
    'User CF Mean BH':kNNMeansUser_cv_results,
    'User CF Mean AH':kNNMeansUser_cv_results_h,
    'Item CF Mean BH':kNNMeansItem_cv_results,
    'Item CF Mean AH':kNNMeansItem_cv_results_h
}

df_list = []
for model_name, result in dplot.items():
    df_temp = pd.DataFrame({
        'test_rmse': result['test_rmse'],
        'test_mae':  result['test_mae']
    })
    df_temp['model'] = model_name
    df_list.append(df_temp)

df_all = pd.concat(df_list, ignore_index=True)

# Calc average per model
means = df_all.groupby('model')[['test_rmse', 'test_mae']].mean().round(4)
means = means.reset_index()

# Melt to long format
plot_df = means.melt(id_vars='model',
                     value_vars=['test_rmse', 'test_mae'],
                     var_name='Metric',
                     value_name='Score')

# Clean names
plot_df['Metric'] = plot_df['Metric'].replace({'test_rmse': 'RMSE', 'test_mae': 'MAE'})

# Order models
model_order = ['User CF Mean BH','User CF Mean AH', 'Item CF Mean BH','Item CF Mean AH', 'SVD BH', 'SVD AH']  
plot_df['model'] = pd.Categorical(plot_df['model'], categories=model_order, ordered=True)
plot_df = plot_df.sort_values('model')

plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.3)

# Lineplot with markers
ax = sns.lineplot(data=plot_df,
                  x='model',
                  y='Score',
                  hue='Metric',
                  marker='o',
                  markersize=14,
                  palette='magma')

# Add values next to each point
for _, row in plot_df.iterrows():
    ax.text(row['model'], 
            row['Score'] + 0.008,
            f'{row["Score"]:.4f}',
            ha='center', va='bottom',
            fontweight='bold', fontsize=12, color='black')

ax.set_title('Average RMSE & MAE From a 5-fold CV\nSmall Dataset',
             fontsize=17, pad=20)
ax.set_ylabel('Error Score', fontsize=13)
ax.set_xlabel('Model', fontsize=13)
ax.legend(title='Metric', title_fontsize=12, fontsize=11, loc='upper right')

plt.ylim(0.60, 0.95)
sns.despine(trim=True)
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Recommender
Best recommendations for everyone function 

In [ ]:
from collections import defaultdict

def get_top_n_for_everyone(predictions, n=10):

    # Map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Sort the predictions for each user and retrieve the best n ones
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n



Fit the model and run the recommender. Top 10 movies

In [ ]:
trainset = data.build_full_trainset()
svd_algo.fit(trainset)
testset = trainset.build_anti_testset()
predictions = svd_algo.test(testset)

top_10 = get_top_n_for_everyone(predictions, n=10)

Add the movie titles to the links dataframe to convert the id to something readable

In [ ]:
df_links2 = df_links.merge(df_meta, left_on='tmdbId',right_on='id').drop(columns=['tmdbId','id'])
df_links2.head()


Show user 2's top recommendations

In [ ]:
movie_titles = dict(zip(df_links2['movieId'], df_links2['title']))

def get_movie_titles(movieIds):
    for i,r in movieIds:
        print(movie_titles[i]," rating: ",r)

get_movie_titles(top_10[2])

### Reload full dataset to calc learning curve for SVD

In [ ]:
# Load the metadata
df_ratings2 = pd.read_csv(data_path+"ratings.csv")
df_ratings2 = df_ratings2[["userId", "movieId", "rating"]]
df_ratings2 = df_ratings2.dropna(subset=["userId", "movieId", "rating"])
df_ratings2["userId"] = df_ratings2["userId"].astype(int) 
df_ratings2["movieId"] = df_ratings2["movieId"].astype(int) 
df_ratings2["rating"] = df_ratings2["rating"].astype(float) 
# If there are duplicate <userId, movieId> pairs, average them. Unlikely tho 
df_ratings2 = df_ratings2.groupby(["userId", "movieId"], as_index=False)["rating"].mean()


df_links2 = pd.read_csv(data_path+"links.csv") 
df_links2 = df_links2[["movieId", "tmdbId"]]
df_links2["tmdbId"] = pd.to_numeric(df_links2["tmdbId"], errors="coerce") 
df_links2 = df_links2.dropna(subset=["tmdbId"]).astype({"tmdbId": int}) 

In [ ]:
# Filter users who have rated too many or too few movies
print("="*50)
print("FILTERING USERS")
print("="*50)

# Count how many movies each user has rated
user_movie_counts = df_ratings2['userId'].value_counts()
print(f"User rating counts - Min: {user_movie_counts.min()}, Max: {user_movie_counts.max()}")

# Remove users who have rated less than 10 movies
min_user_ratings = 10
users_to_keep = user_movie_counts[user_movie_counts >= min_user_ratings].index
print(f"Users with at least {min_user_ratings} ratings: {len(users_to_keep)} out of {len(user_movie_counts)}")

# Remove users who rated too many movies (top 5%)
max_percentile = 95
max_user_ratings = np.percentile(user_movie_counts, max_percentile)
users_to_keep = users_to_keep[user_movie_counts[users_to_keep] <= max_user_ratings]
print(f"Users after removing top 5% (>{max_user_ratings:.0f} ratings): {len(users_to_keep)}")

# Filter the dataset to keep only selected users
ratings_filtered2 = df_ratings2[df_ratings2['userId'].isin(users_to_keep)].copy()
print(f"Ratings after user filtering: {ratings_filtered2.shape}")


In [ ]:
# Filter movies with less than 20 ratings

print("="*50)
print("FILTERING MOVIES")
print("="*50)

# Count how many ratings each movie has received
movie_rating_counts = ratings_filtered2['movieId'].value_counts()
print(f"Movie rating counts - Min: {movie_rating_counts.min()}, Max: {movie_rating_counts.max()}")

# Keep only movies with at least 20 ratings
min_movie_ratings = 20
movies_to_keep = movie_rating_counts[movie_rating_counts >= min_movie_ratings].index
print(f"Movies with at least {min_movie_ratings} ratings: {len(movies_to_keep)} out of {len(movie_rating_counts)}")

# Filter the dataset to keep only selected movies
ratings_final2 = ratings_filtered2[ratings_filtered2['movieId'].isin(movies_to_keep)].copy()
print(f"Final ratings shape: {ratings_final2.shape}")


In [ ]:
# Create index mappings
print("="*50)
print("CREATING INDEX MAPPINGS")
print("="*50)

unique_users = sorted(ratings_final2['userId'].unique())
unique_movies = sorted(ratings_final2['movieId'].unique())

print(f"Final number of users: {len(unique_users)}")
print(f"Final number of movies: {len(unique_movies)}")

user_id_to_index = {user_id: idx for idx, user_id in enumerate(unique_users)}
index_to_user_id = {idx: user_id for user_id, idx in user_id_to_index.items()}
movie_id_to_index = {movie_id: idx for idx, movie_id in enumerate(unique_movies)}
index_to_movie_id = {idx: movie_id for movie_id, idx in movie_id_to_index.items()}

ratings_final2['user_idx'] = ratings_final2['userId'].map(user_id_to_index)
ratings_final2['movie_idx'] = ratings_final2['movieId'].map(movie_id_to_index)

In [ ]:
# Create the user-movie rating matrix
print("="*50)
print("CREATING USER-MOVIE MATRIX")
print("="*50)

from scipy.sparse import csr_matrix

n_users = len(unique_users)
n_movies = len(unique_movies)

# Convert to numpy arrays for faster indexing
user_indices = ratings_final2['user_idx'].values.astype(int)
movie_indices = ratings_final2['movie_idx'].values.astype(int)
rating_values = ratings_final2['rating'].values.astype(float)

# Create a sparse user-movie matrix using csr_matrix
# Data, (row_indices, col_indices)
user_movie_matrix2 = csr_matrix((rating_values, (user_indices, movie_indices)), shape=(n_users, n_movies))

print(f"User-movie matrix shape: {user_movie_matrix2.shape}")
# Sparsity for csr_matrix can be calculated as 1 - (number of stored elements / total possible elements)
print(f"Matrix sparsity: {(1 - user_movie_matrix2.nnz / (n_users * n_movies)) * 100:.2f}%")

In [ ]:
print("Loading data into Surprise format...")
reader2 = Reader(rating_scale=(0.5, 5.0))
data2 = Dataset.load_from_df(ratings_final2[['userId', 'movieId', 'rating']], reader2)
print("Data loaded successfully into Surprise Dataset.")

In [ ]:
# Split and keep a fixed test set
trainset, testset = train_test_split(data2, test_size=0.2, random_state=42)

# Convert global trainset to list of ratings for sampling
all_ratings = [(trainset.to_raw_uid(u), trainset.to_raw_iid(i), r) 
               for u, i, r in trainset.all_ratings()]

# Learning curve subsets
train_fractions = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
train_rmses = []
test_rmses  = []

print("Computing learning curve")
for frac in train_fractions:
    print(f"  → Using {frac*100:.0f}% of training data ({int(frac*len(all_ratings)):,} ratings)")

    # Sample subset of ratings
    subset_size = int(frac * len(all_ratings))
    subset_ratings = np.random.choice(len(all_ratings), size=subset_size, replace=False)
    subset_list = [all_ratings[i] for i in subset_ratings]

    # Build a new Trainset from subset
    subset_data = Dataset.load_from_df(
        pd.DataFrame(subset_list, columns=['user_id', 'item_id', 'rating']),
        reader
    )
    subset_trainset = subset_data.build_full_trainset()

    # Train model using hypertuned SVD model we made earlier
    model = svd_algo
    model.fit(subset_trainset)

    # Train rmse
    train_predictions = model.test(subset_trainset.build_testset())
    train_rmse = accuracy.rmse(train_predictions, verbose=False)
    train_rmses.append(train_rmse)

    # Test rmse
    test_predictions = model.test(testset)
    test_rmse = accuracy.rmse(test_predictions, verbose=False)
    test_rmses.append(test_rmse)



Plot our result

In [ ]:
plt.figure(figsize=(14, 9))

# Red blue values
plt.plot(np.array(train_fractions)*100, train_rmses, 'o-', 
         label='Training RMSE', color='#3498db', linewidth=2, markersize=10)
plt.plot(np.array(train_fractions)*100, test_rmses, 'o-', 
         label='Test RMSE', color='#e74c3c', linewidth=2, markersize=10)

# Add value labels
for x, y_tr, y_te in zip(np.array(train_fractions)*100, train_rmses, test_rmses):
    plt.text(x, y_tr - 0.005, f'{y_tr:.3f}', ha='center', va='top', color='#2980b9')
    plt.text(x, y_te + 0.005, f'{y_te:.3f}', ha='center', va='bottom', color='darkred')

plt.title('SVD Learning Curve', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Training Data Used (%)', fontsize=13)
plt.ylabel('RMSE', fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
# plt.ylim(0.80, 0.95)
sns.despine(trim=True)
plt.tight_layout()
plt.show()